In [1]:
# ============================================================================
# 04 — RUN THE MODELS
# ----------------------------------------------------------------------------
# One question per API call. Every attempt is written to disk, including
# failures — a call that vanishes without a trace is the bug you cannot find
# afterwards. Resumable: re-running skips work already done.
# ============================================================================
import os, json, time, threading, hashlib, requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import pandas as pd, numpy as np
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent/".env")
KEY = os.environ.get("OPENROUTER_API_KEY", "")
assert len(KEY) > 20, "OPENROUTER_API_KEY missing from .env"
ENDPOINT = "https://openrouter.ai/api/v1/chat/completions"

ROOT = Path.cwd().parent
PROC, RES = ROOT/"data"/"processed", ROOT/"results"
RES.mkdir(parents=True, exist_ok=True)
VERSION = "v2"

# ---- load benchmark and prompts -----------------------------------------
FULL = pd.read_json(PROC/f"sighabit_{VERSION}_full.jsonl", lines=True)
USUAL = pd.read_json(PROC/f"sighabit_{VERSION}_usually.jsonl", lines=True)
PR = json.loads((PROC/f"prompts_{VERSION}.json").read_text())
SHA = hashlib.sha256((PROC/f"sighabit_{VERSION}_full.jsonl").read_bytes()).hexdigest()[:16]
print(f"benchmark {len(FULL)} + {len(USUAL)} usually  sha={SHA}")

def render(item, swap=False):
    if item.get("condition") == "usually":
        body = (f"Across its known direct targets, what is the net regulatory "
                f"effect of {item['source']} at the protein level?\n{PR['guide_usually']}")
    else:
        body = (f"What is the net regulatory effect of {item['source']} on "
                f"{item['target']} at the protein level?\n{PR['guide']}")
    return f"{body}\n\nRespond with JSON only:\n" + (
        PR['schema_swapped'] if swap else PR['schema'])

# ---- parser --------------------------------------------------------------
import re
SYN = {"stimulates":"stimulates","stimulate":"stimulates","activates":"stimulates",
 "activate":"stimulates","activation":"stimulates","positive":"stimulates",
 "upregulates":"stimulates","induces":"stimulates",
 "inhibits":"inhibits","inhibit":"inhibits","inhibition":"inhibits",
 "represses":"inhibits","repress":"inhibits","negative":"inhibits",
 "downregulates":"inhibits","suppresses":"inhibits",
 "not_established":"not_established","not established":"not_established",
 "unknown":"not_established","none":"not_established","unclear":"not_established"}

def strip_reasoning(t):
    """Some models restate the schema inside their scratchpad. Remove it, or
    we would parse the example instead of the answer."""
    t = re.sub(r"<think>.*?</think>", " ", t, flags=re.S|re.I)
    t = re.sub(r"^.*?</think>", " ", t, flags=re.S|re.I)
    return t.strip() or t

def parse(text):
    if not isinstance(text,str) or not text.strip(): return None,"failed"
    b = strip_reasoning(text)
    try:
        o = json.loads(b)
        if isinstance(o,dict): return o,"strict"
    except Exception: pass
    c = re.sub(r"```(?:json)?|```", " ", b)
    for blob in reversed(re.findall(r"\{[^{}]*\}", c, flags=re.S)):
        try:
            o = json.loads(re.sub(r",\s*([}\]])", r"\1", blob))
            if isinstance(o,dict) and "answer" in o: return o,"repaired"
        except Exception: continue
    m = re.search(r'"?answer"?\s*[:=]\s*"?([A-Za-z_ ]+?)"?\s*[,\n}]', c, re.I)
    if m:
        out = {"answer": m.group(1).strip()}
        cf = re.search(r'"?confidence"?\s*[:=]\s*(\d+)', c, re.I)
        if cf: out["confidence"] = int(cf.group(1))
        return out,"regex"
    return None,"failed"

def normalize(p):
    if not p or "answer" not in p: return None
    return SYN.get(str(p["answer"]).strip().lower().replace("-","_"))

# quick self-test — never trust a parser you have not tested
for txt, want in [('{"answer":"stimulates","confidence":8}',"stimulates"),
                  ('```json\n{"answer":"inhibits",}\n```',"inhibits"),
                  ('<think>maybe stimulates</think>{"answer":"inhibits"}',"inhibits"),
                  ('{"answer":"activates"}',"stimulates"),
                  ('I think it activates.',None)]:
    got = normalize(parse(txt)[0])
    assert got == want, f"parser failed: {txt!r} -> {got}, wanted {want}"
print("parser self-test: all pass")

# ---- models --------------------------------------------------------------
MODELS = [
 ("gpt-oss-120b",    "openai/gpt-oss-120b",        0.03,  0.17, 6000),
 ("qwen3-32b",       "qwen/qwen3-32b",             0.10,  0.30, 4000),
 ("gemini-3.7-flash","google/gemini-3.7-flash",    0.1875,0.9375,3000),
 ("gpt-5.6-luna",    "openai/gpt-5.6-luna",        0.10,  0.60, 5000),
 ("gpt-5.6-terra",   "openai/gpt-5.6-terra",       1.00,  6.00, 5000),
]
BUDGET, WORKERS, TEMP = 5.00, 6, 0.3
SPEND = {"usd":0.0}; STOP = threading.Event(); LOCK = threading.Lock()

def call(mid, user, max_tok, web=False, timeout=180):
    body = {"model":mid, "temperature":TEMP, "max_tokens":max_tok,
            "messages":[{"role":"system","content":PR["system"]},
                        {"role":"user","content":user}]}
    if web: body["plugins"] = [{"id":"web","max_results":3}]
    for att in range(4):
        if STOP.is_set(): return dict(text=None,usage={},http=0,error="stopped",prov=None)
        try:
            r = requests.post(ENDPOINT, headers={"Authorization":f"Bearer {KEY}"},
                              json=body, timeout=timeout)
            if r.status_code == 200:
                d = r.json(); ch = d["choices"][0]
                txt = ch["message"].get("content")
                if not txt:
                    return dict(text=None, usage=d.get("usage",{}), http=204,
                                error=f"empty ({ch.get('finish_reason')})",
                                prov=d.get("provider"))
                return dict(text=txt, usage=d.get("usage",{}), http=200,
                            error=None, prov=d.get("provider"))
            if r.status_code in (429,503,529):
                time.sleep(min(5*(1.8**att),60)); continue
            if r.status_code in (401,402,403):
                STOP.set()
                return dict(text=None,usage={},http=r.status_code,error=r.text[:200],prov=None)
            return dict(text=None,usage={},http=r.status_code,error=r.text[:200],prov=None)
        except Exception:
            time.sleep(min(2**att,20))
    return dict(text=None,usage={},http=0,error="retries exhausted",prov=None)

def run(name, mid, pin, pout, max_tok, items, arm, swap=False, web=False):
    tag  = f"{VERSION}__{name}__{arm}"
    path = RES/f"{tag}.jsonl"
    done = set()
    if path.exists():
        rows = [json.loads(l) for l in path.open(encoding="utf-8") if l.strip()]
        done = {r["item_id"] for r in rows if r.get("http")==200 and r.get("raw")}
    todo = items[~items.item_id.isin(done)]
    if not len(todo):
        print(f"  {tag}: complete ({len(done)})"); return
    print(f"  {tag}: {len(todo)} to do", flush=True)
    t0, n, bad = time.time(), [0], [0]
    def one(d, f):
        if STOP.is_set(): return
        res = call(mid, render(d, swap), max_tok, web)
        parsed, tier = parse(res["text"]) if res["http"]==200 else (None,"failed")
        u = res.get("usage") or {}
        cost = u.get("prompt_tokens",0)/1e6*pin + u.get("completion_tokens",0)/1e6*pout
        rec = dict(item_id=d["item_id"], benchmark_sha=SHA, model=name, arm=arm,
            model_id=mid, provider=res["prov"], swap=bool(swap), web=bool(web),
            source=d["source"], target=d.get("target"),
            condition=d["condition"], truth=d["answer"],
            protein_usually=d.get("protein_usually"), group=d.get("group"),
            n_refs=d.get("n_refs"), is_decoy=bool(d.get("is_decoy", False)),
            pred=normalize(parsed), confidence=(parsed or {}).get("confidence"),
            rationale=(parsed or {}).get("rationale"), parse_tier=tier,
            raw=res["text"], prompt_tokens=u.get("prompt_tokens"),
            completion_tokens=u.get("completion_tokens"), cost_usd=cost,
            http=res["http"], error=res["error"])
        with LOCK:
            SPEND["usd"] += cost; n[0] += 1
            if res["http"] != 200: bad[0] += 1
            f.write(json.dumps(rec)+"\n"); f.flush()
            if SPEND["usd"] > BUDGET:
                STOP.set(); print(f"\n*** BUDGET ${BUDGET} REACHED — STOPPING ***")
            if n[0] % max(1,len(todo)//5) == 0 or n[0]==len(todo):
                el = (time.time()-t0)/60
                print(f"    {n[0]}/{len(todo)}  {el:.1f}m  fail {bad[0]}  "
                      f"${SPEND['usd']:.3f}", flush=True)
    with open(path,"a",encoding="utf-8") as f:
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            list(ex.map(lambda r: one(r[1].to_dict(), f), todo.iterrows()))
    print(f"    done {(time.time()-t0)/60:.1f}m  failures {bad[0]}/{len(todo)}")

# ---- the plan ------------------------------------------------------------
print("\n"+"="*70); print("MAIN ARM — all models, all questions"); print("="*70)
for name, mid, pin, pout, mt in MODELS:
    if STOP.is_set(): break
    print(f"\n{name}")
    run(name, mid, pin, pout, mt, FULL,  "main")
    run(name, mid, pin, pout, mt, USUAL, "usually")

print("\n"+"="*70); print("ORDER-SWAP ARM — validity check"); print("="*70)
MAIN_ONLY = FULL[~FULL.is_decoy]
for name, mid, pin, pout, mt in MODELS[:3]:
    if STOP.is_set(): break
    run(name, mid, pin, pout, mt, MAIN_ONLY, "swap", swap=True)

print("\n"+"="*70); print("WEB-SEARCH ARM — Gemini"); print("="*70)
if not STOP.is_set():
    run("gemini-3.7-flash","google/gemini-3.7-flash",.1875,.9375,3000,
        FULL, "web", web=True)

# ---- inventory -----------------------------------------------------------
print("\n"+"="*70); print("INVENTORY"); print("="*70)
A = pd.concat([pd.read_json(p, lines=True) for p in RES.glob("*.jsonl")],
              ignore_index=True)
print(f"rows {len(A):,}   spent ${SPEND['usd']:.2f}")
print(f"http failures {(A.http!=200).mean():.2%}   "
      f"unparsed {(A[A.http==200].pred.isna()).mean():.2%}")
print("\nitems per model x arm:")
print(A[A.http==200].pivot_table(index="model", columns="arm",
      values="item_id", aggfunc="nunique", fill_value=0).to_string())
if (A.http!=200).any():
    print("\nfailures:"); print(A[A.http!=200].groupby(["model","http"]).size().to_string())

AssertionError: OPENROUTER_API_KEY missing from .env